# 🧠 Notebook 03 — Model Training: Neural Network + XGBoost
## Full Training Pipeline with Cross-Validation & Ensemble

This notebook trains and validates both the neural network and XGBoost
models, performs K-fold cross-validation, and builds the optimised ensemble.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers
import xgboost as xgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
import warnings, sys, os
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))
os.makedirs("../models/saved", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)
tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow {tf.__version__} | XGBoost {xgb.__version__}")


## 📂 Load & Prepare Data

In [ ]:
# ── Regenerate data (or load from CSV if available) ─────────────────────
DATA_PATH = "../data/raw/cdr_credit_data.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded from CSV: {df.shape}")
else:
    exec(open('../src/data/synthetic_cdr_generator.py').read())
    gen = CDRCreditDataGenerator(n_subscribers=50_000, random_seed=42)
    df = gen.generate()
    df.to_csv(DATA_PATH, index=False)
    print(f"Generated and saved: {df.shape}")

FEATURE_COLS = [c for c in df.columns if c not in ["subscriber_id","latent_creditworthiness","default_flag"]]
X = df[FEATURE_COLS].values.astype(np.float32)
y = df["default_flag"].values

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

scaler = RobustScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

n_neg, n_pos = (y_train==0).sum(), (y_train==1).sum()
class_weight = {0: 1.0, 1: round(n_neg/n_pos, 2)}
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print(f"Default rate: {y_train.mean():.2%} | Class weight: {class_weight}")


## 🧠 Neural Network Architecture

In [ ]:
def build_model(input_dim, l1=1e-4, l2=1e-3):
    reg = regularizers.L1L2(l1=l1, l2=l2)
    inp = keras.Input(shape=(input_dim,), name="cdr_features")
    x = layers.BatchNormalization()(inp)
    x = layers.Dropout(0.05)(x)
    
    specs = [(256, 0.30, True), (128, 0.25, True), (64, 0.20, False), (32, 0.0, False)]
    for i, (units, drop, bn) in enumerate(specs):
        x = layers.Dense(units, activation="gelu", kernel_regularizer=reg, name=f"d{i+1}")(x)
        if bn: x = layers.BatchNormalization()(x)
        if drop > 0: x = layers.Dropout(drop)(x)
    
    out = layers.Dense(1, activation="sigmoid", name="prob")(x)
    
    lr = keras.optimizers.schedules.CosineDecay(0.001, decay_steps=5000, alpha=1e-4)
    model = keras.Model(inp, out, name="CDR_CreditScorer_v1")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc"), "precision", "recall"]
    )
    return model

model = build_model(X_train_sc.shape[1])
model.summary()
total_params = model.count_params()
print(f"\n  Total Parameters: {total_params:,}")
print(f"  Architecture: Input({X_train_sc.shape[1]}) → BN → 256 → 128 → 64 → 32 → 1")


## 🏋️ Model Training

In [ ]:
cb_list = [
    callbacks.EarlyStopping(monitor="val_auc", patience=15, mode="max",
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_auc", factor=0.5, patience=8,
                                mode="max", verbose=1, min_lr=1e-6),
    callbacks.ModelCheckpoint("../models/saved/best_nn.keras",
                               monitor="val_auc", save_best_only=True, mode="max", verbose=0)
]

print("🚀 Training neural network...")
history = model.fit(
    X_train_sc, y_train,
    validation_data=(X_val_sc, y_val),
    epochs=80, batch_size=512,
    class_weight=class_weight,
    callbacks=cb_list, verbose=1
)

best_val_auc = max(history.history["val_auc"])
best_epoch   = history.history["val_auc"].index(best_val_auc) + 1
print(f"\n✅ Training complete!")
print(f"   Best Val AUC: {best_val_auc:.4f} at epoch {best_epoch}")


## 🌲 XGBoost Training

In [ ]:
print("🌲 Training XGBoost model...")
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, gamma=0.1,
    scale_pos_weight=round(n_neg/n_pos, 1),
    eval_metric="auc", early_stopping_rounds=30,
    random_state=42, n_jobs=-1, tree_method="hist"
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)
xgb_val_auc = roc_auc_score(y_val, xgb_model.predict_proba(X_val)[:,1])
print(f"\n✅ XGBoost trained | Best Val AUC: {xgb_val_auc:.4f}")


## 🔀 Ensemble & Test Evaluation

In [ ]:
# Test set predictions
nn_test  = model.predict(X_test_sc, verbose=0).flatten()
xgb_test = xgb_model.predict_proba(X_test)[:,1]
ens_test = 0.55 * nn_test + 0.45 * xgb_test

def eval_metrics(y_true, y_score, name):
    auc = roc_auc_score(y_true, y_score)
    gini = 2*auc - 1
    ks, _ = ks_2samp(y_score[y_true==0], y_score[y_true==1])
    n = len(y_true); top20 = int(n*0.20)
    bc20 = y_true[np.argsort(y_score)[::-1][:top20]].sum() / y_true.sum()
    return {"Model": name, "AUC": round(auc,4), "Gini": round(gini,4),
            "KS": round(ks,4), "Bad Capture @20%": round(bc20,4)}

results = pd.DataFrame([
    eval_metrics(y_test, nn_test, "Neural Network (CDR)"),
    eval_metrics(y_test, xgb_test, "XGBoost (CDR)"),
    eval_metrics(y_test, ens_test, "Ensemble ★"),
])
print("\n" + "="*65)
print("TEST SET RESULTS")
print("="*65)
print(results.to_string(index=False))
print("\n✅ Recommended model: Ensemble (highest AUC)")

# Save results
results.to_csv("../reports/metrics/model_results.csv", index=False)
import joblib; joblib.dump(scaler, "../models/saved/scaler.pkl")
print("\nModels & results saved.")


## 📊 5-Fold Cross-Validation

In [ ]:
print("Running 5-fold stratified cross-validation...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (tr_idx, vl_idx) in enumerate(skf.split(X, y)):
    X_tr, X_vl = X[tr_idx], X[vl_idx]
    y_tr, y_vl = y[tr_idx], y[vl_idx]
    sc = RobustScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_vl_sc = sc.transform(X_vl)
    
    fold_model = build_model(X_tr_sc.shape[1])
    fold_model.fit(X_tr_sc, y_tr, validation_data=(X_vl_sc, y_vl),
                   epochs=40, batch_size=512, class_weight=class_weight,
                   callbacks=[callbacks.EarlyStopping(monitor="val_auc", patience=8, 
                               mode="max", restore_best_weights=True)],
                   verbose=0)
    
    pred = fold_model.predict(X_vl_sc, verbose=0).flatten()
    auc  = roc_auc_score(y_vl, pred)
    gini = 2*auc - 1
    ks, _ = ks_2samp(pred[y_vl==0], pred[y_vl==1])
    fold_results.append({"Fold": fold+1, "AUC": round(auc,4), "Gini": round(gini,4), "KS": round(ks,4)})
    print(f"  Fold {fold+1}: AUC={auc:.4f} | Gini={gini:.4f} | KS={ks:.4f}")

cv_df = pd.DataFrame(fold_results)
print("\nCross-Validation Summary:")
print(cv_df.describe().round(4)[["AUC","Gini","KS"]].to_string())
print(f"\nMean AUC: {cv_df['AUC'].mean():.4f} ± {cv_df['AUC'].std():.4f}")
